# 05 - Retrieval Quality Evaluation

| Metric | Measures |
|---|---|
| **Hit@K** | % of queries where relevant chunk is in top-K |
| **MRR@K** | Mean Reciprocal Rank |
| **NDCG@K** | Position-weighted ranking quality |

**Strategy**: use each chunk's first 250 chars as the query, check if original chunk is retrieved in top-K.

In [1]:
import os, sys, random
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

HF_API_KEY       = os.getenv('HF_API_KEY')
COHERE_API_KEY   = os.getenv('COHERE_API_KEY', '')
WEAVIATE_URL     = os.getenv('WEAVIATE_URL', '').rstrip('/')
WEAVIATE_API_KEY = os.getenv('WEAVIATE_API_KEY')
COLLECTION       = 'ScmChunks'

sys.path.insert(0, str(Path('..').resolve()))
from tools import eval_retrieval

SAMPLE_SIZE = 20
K_VALUES    = [5, 10]
print(f'Sample: {SAMPLE_SIZE}  K: {K_VALUES}')

Sample: 20  K: [5, 10]


## 1. Sample chunks

In [2]:
import weaviate
from weaviate.classes.init import Auth, AdditionalConfig, Timeout

_headers = {}
if HF_API_KEY:
    _headers['X-HuggingFace-Api-Key'] = HF_API_KEY.strip()
if COHERE_API_KEY:
    _headers['X-Cohere-Api-Key'] = COHERE_API_KEY.strip()

client = weaviate.connect_to_weaviate_cloud(
    cluster_url      = WEAVIATE_URL,
    auth_credentials = Auth.api_key(WEAVIATE_API_KEY),
    headers          = _headers,
    additional_config= AdditionalConfig(timeout=Timeout(init=60, query=30, insert=60)),
    skip_init_checks = True,
)
collection  = client.collections.get(COLLECTION)
print(f'Total: {collection.aggregate.over_all(total_count=True).total_count}')

all_chunks = [obj.properties
              for obj in collection.iterator(return_properties=['chunk_id','text','source_file'])
              if len(obj.properties.get('text','')) > 100]
client.close()

random.seed(42)
sample = random.sample(all_chunks, min(SAMPLE_SIZE, len(all_chunks)))
print(f'Sampled {len(sample)} chunks')

Total: 769
Sampled 20 chunks


## 2. Vector only

In [3]:
print('Vector-only...')
vec = {}
for k in K_VALUES:
    m = eval_retrieval(sample, k=k, use_rerank=False)
    vec[k] = m; print(f'  K={k}: {m}')

Vector-only...
  K=5: {'Hit@5': 0.9, 'MRR@5': 0.8667, 'NDCG@5': 0.875, 'n_queries': 20}
  K=10: {'Hit@10': 0.95, 'MRR@10': 0.8738, 'NDCG@10': 0.8917, 'n_queries': 20}


## 3. Vector + Cohere reranker

In [4]:
print('Vector + rerank...')
rnk = {}
for k in K_VALUES:
    m = eval_retrieval(sample, k=k, use_rerank=True, fetch_limit=20)
    rnk[k] = m; print(f'  K={k}: {m}')

Vector + rerank...
  warn: skipping chunk 10 — WeaviateQueryError
  warn: skipping chunk 11 — WeaviateQueryError
  warn: skipping chunk 12 — WeaviateQueryError
  warn: skipping chunk 13 — WeaviateQueryError
  warn: skipping chunk 14 — WeaviateQueryError
  warn: skipping chunk 15 — WeaviateQueryError
  warn: skipping chunk 16 — WeaviateQueryError
  warn: skipping chunk 17 — WeaviateQueryError
  warn: skipping chunk 18 — WeaviateQueryError
  warn: skipping chunk 19 — WeaviateQueryError
  K=5: {'Hit@5': 0.5, 'MRR@5': 0.5, 'NDCG@5': 0.5, 'n_queries': 20}
  warn: skipping chunk 0 — WeaviateQueryError
  warn: skipping chunk 1 — WeaviateQueryError
  warn: skipping chunk 2 — WeaviateQueryError
  warn: skipping chunk 3 — WeaviateQueryError
  warn: skipping chunk 4 — WeaviateQueryError
  warn: skipping chunk 5 — WeaviateQueryError
  warn: skipping chunk 6 — WeaviateQueryError
  warn: skipping chunk 7 — WeaviateQueryError
  warn: skipping chunk 8 — WeaviateQueryError
  warn: skipping chunk 9 — We

## 4. Comparison table

In [5]:
import pandas as pd

rows = []
for k in K_VALUES:
    for metric in [f'Hit@{k}', f'MRR@{k}', f'NDCG@{k}']:
        v, r = vec[k][metric], rnk[k][metric]
        delta = r - v
        rows.append({'Metric': metric, 'Vector only': v, 'Vector + Rerank': r,
                     'Delta': round(delta,4),
                     'Improvement': f'+{delta*100:.1f}%' if delta>=0 else f'{delta*100:.1f}%'})
print(pd.DataFrame(rows).to_string(index=False))

 Metric  Vector only  Vector + Rerank   Delta Improvement
  Hit@5       0.9000              0.5 -0.4000      -40.0%
  MRR@5       0.8667              0.5 -0.3667      -36.7%
 NDCG@5       0.8750              0.5 -0.3750      -37.5%
 Hit@10       0.9500              0.0 -0.9500      -95.0%
 MRR@10       0.8738              0.0 -0.8738      -87.4%
NDCG@10       0.8917              0.0 -0.8917      -89.2%


## 5. Per-source breakdown (Hit@5)

In [6]:
from collections import defaultdict
from weaviate.classes.query import Rerank

_headers2 = {}
if HF_API_KEY:
    _headers2['X-HuggingFace-Api-Key'] = HF_API_KEY.strip()
if COHERE_API_KEY:
    _headers2['X-Cohere-Api-Key'] = COHERE_API_KEY.strip()

client = weaviate.connect_to_weaviate_cloud(
    cluster_url      = WEAVIATE_URL,
    auth_credentials = Auth.api_key(WEAVIATE_API_KEY),
    headers          = _headers2,
    additional_config= AdditionalConfig(timeout=Timeout(init=60, query=30, insert=60)),
    skip_init_checks = True,
)
collection = client.collections.get(COLLECTION)

K=5
hv, hr = defaultdict(list), defaultdict(list)
for chunk in sample:
    src, q, rid = chunk.get('source_file','?'), chunk['text'][:250], chunk['chunk_id']
    rv = collection.query.near_text(query=q, limit=K, return_properties=['chunk_id'])
    hv[src].append(float(rid in [o.properties.get('chunk_id','') for o in rv.objects][:K]))
    try:
        rr = collection.query.near_text(query=q, limit=20,
                                         rerank=Rerank(prop='text', query=q),
                                         return_properties=['chunk_id'])
        hr[src].append(float(rid in [o.properties.get('chunk_id','') for o in rr.objects][:K]))
    except Exception:
        hr[src].append(0.0)
client.close()

print(f'Hit@{K} by source:')
print(f'  {"Source":<35} {"Vector":>8} {"+ Rerank":>10} {"Delta":>8}')
print('  '+'-'*63)
for src in sorted(hv):
    v=sum(hv[src])/len(hv[src]); r=sum(hr[src])/len(hr[src]); d=r-v
    print(f'  {src:<35} {v:>8.3f} {r:>10.3f} {"+" if d>=0 else ""}{round(d,3):>7}')

Hit@5 by source:
  Source                                Vector   + Rerank    Delta
  ---------------------------------------------------------------
  2307.03875v2.pdf                       1.000      0.500    -0.5
  2504.03692v1.pdf                       0.800      0.500    -0.3
  2508.21622v1.pdf                       1.000      0.250   -0.75
  supply_chain_management_tutorial.pdf    1.000      0.750   -0.25
